# Promo Finder: Kuku Problem 1

Reads a folder of microdrama screenplays (one `.txt` per episode) and finds 20 to 60 second promo clips that **open on a hook**, **end on a cliffhanger** and **make sense on their own**.

How the work is split:
1. **Code** parses each screenplay into numbered, timed lines (regex).
2. **AI** reads one episode at a time and picks up to 3 promos as *line numbers*: the hook line where the clip starts and the cliffhanger line where it cuts.
3. **Code** turns line numbers into timestamps and checks the rules (20 to 60s, real lines). If a rule fails, the model is told what's wrong and gets one chance to fix it.
4. **Code** ranks the promos and removes overlaps. Then a **cold-viewer test** (AI) reads each shortlisted clip *without any context*, like a stranger scrolling, and says whether it makes sense and what question the ending leaves.
5. **A human** checks the top 10 before anything is posted (last section).

To run on a new show, change `FOLDER` and Run All. To swap the model, change `BASE_URL`, `API_KEY` and `MODEL`.

In [ ]:

import os, re, json, time
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, ValidationError

FOLDER = "../I Married a Broken Soldier. He was a Duke"

BASE_URL = os.getenv("BASE_URL", "http://localhost:11434/v1")
API_KEY = os.getenv("API_KEY", "ollama")
MODEL = os.getenv("MODEL", "qwen2.5:7b")
JUDGE_MODEL = os.getenv("JUDGE_MODEL", MODEL)


USE_CACHE = True
PAUSE = 15

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9.-]+", "_", s).strip("_")

SHOW = os.path.basename(os.path.normpath(FOLDER))
OUT = os.path.join("outputs", safe_name(SHOW), safe_name(MODEL))
os.makedirs(os.path.join(OUT, "raw"), exist_ok=True)
print("show:", SHOW)
print("model:", MODEL, "| judge:", JUDGE_MODEL, "| outputs go to", OUT)

## 1. Parse the screenplays (code)

Every block in the script looks like this:
```
**NAME (Emotion):**
Start: (00:11.30) | End: (00:13.10)
"The line of dialogue."
```

In [ ]:
def to_seconds(t):
    mins, secs = t.split(":")
    return int(mins) * 60 + float(secs)

def mmss(sec):
    return f"{int(sec // 60):02d}:{sec % 60:05.2f}"

def parse_episode(path):
    text = open(path, encoding="utf-8").read()
    rows = text.split("\n")

    number = re.search(r"EPISODE NUMBER:\s*Episode\s*(\d+)", text, re.I)
    logline = re.search(r"EPISODE LOGLINE:\s*(.*)", text, re.I)
    ep = {
        "number": int(number.group(1)) if number else None,
        "logline": logline.group(1).strip() if logline else "",
        "lines": [],
    }

    for i, row in enumerate(rows):
        m = re.match(r"Start: \((\d+:\d+\.\d+)\) \| End: \((\d+:\d+\.\d+)\)", row.strip())
        if not m:
            continue
        head = rows[i - 1].strip().strip("*").rstrip(":")
        speaker = head.split("(")[0].strip()
        note = ", ".join(re.findall(r"\((.*?)\)", head))
        if speaker in ("CAMERA", "ACTION", "SOUND FX"):
            kind = speaker
        else:
            kind = "DIALOGUE"
        ep["lines"].append({
            "start": to_seconds(m.group(1)),
            "end": to_seconds(m.group(2)),
            "kind": kind,
            "speaker": speaker,
            "note": note,
            "text": rows[i + 1].strip() if i + 1 < len(rows) else "",
        })

    ep["length"] = max(line["end"] for line in ep["lines"])
    return ep

files = sorted(f for f in os.listdir(FOLDER) if f.endswith(".txt"))
episodes = []
for i, name in enumerate(files):
    ep = parse_episode(os.path.join(FOLDER, name))
    if ep["number"] is None:
        ep["number"] = i + 1
    episodes.append(ep)

print(len(episodes), "episodes,", sum(len(ep["lines"]) for ep in episodes), "timed lines")
print("episode 1:", len(episodes[0]["lines"]), "lines,", mmss(episodes[0]["length"]), "long")

## 2. Shrink each episode for the model (code)

CAMERA lines are about half the script, and the story is in the dialogue and action. Dropping them keeps each episode under about 2k tokens, which small local models can handle. Lines are numbered so the model can point at them.

In [ ]:
def story_lines(ep):
    return [line for line in ep["lines"] if line["kind"] != "CAMERA"]

def line_text(line):
    if line["kind"] == "DIALOGUE":
        return f"{line['speaker']} ({line['note']}): {line['text']}"
    return f"{line['kind']}: {line['text']}"

def shrink(ep):
    out = []
    for n, line in enumerate(story_lines(ep), start=1):
        out.append(f"L{n} [{line['start']:.1f}-{line['end']:.1f}] {line_text(line)}")
    return "\n".join(out)

print(shrink(episodes[0])[:700])

## 3. The model's job (AI)

The model only picks **which lines**: its judgement on hook, cliffhanger and whether it makes sense. Code does all the time maths. (The first version asked the model for timestamps directly. It got the maths wrong on most clips; see the README.)

In [ ]:
class Segment(BaseModel):
    from_line: int
    to_line: int

class Promo(BaseModel):
    segments: list[Segment]
    why: str
    hook_score: int
    cliff_score: int
    standalone_score: int

class EpisodePromos(BaseModel):
    promos: list[Promo]

PROMPT = """You pick promo clips for short vertical drama series (like ReelShort or DramaBox).
A promo is a 20 to 60 second clip from the show, posted on TikTok, Instagram or YouTube to pull in NEW viewers who have never seen the show.

You get one episode's screenplay. Every line has a number (L1, L2, ...) and [start-end] time in seconds.
Find 3 promos in this episode, each from a different moment. Return fewer only if the episode really has fewer good moments.
Give each promo as segments of lines, from_line to to_line.
- from_line of the first segment is the HOOK. The clip starts exactly on this line, so it must make a scrolling viewer stop in the first 3 seconds: a slap, a reveal, a shocking line, a threat. Not a calm line, and not a confused question like "What do you mean?".
- to_line of the last segment is the CLIFFHANGER. The clip cuts right after this line, so it must be unresolved: a threat, a question, a reveal before anyone reacts. Never end on a calm, resolved or quiet moment (someone breathing, smiling, walking away, music fading).
- Length = end time of to_line minus start time of from_line. It MUST be between 20 and 60 seconds.
- Usually use one segment. You may join up to 3 segments from this episode, in time order, if skipping a slow part makes a stronger promo.
- It must make sense to a stranger who has not seen earlier episodes.
- Scores are 1 to 10. Be strict: 9 or 10 only for moments that would really stop a scroll.

Reply with JSON only, in exactly this shape:
{"promos": [{"segments": [{"from_line": 3, "to_line": 21}], "why": "one line on why it works", "hook_score": 7, "cliff_score": 8, "standalone_score": 6}]}
If the episode has no good promo, reply {"promos": []}."""

def ask_llm(messages, model=None):
    reply = client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    time.sleep(PAUSE)
    return reply.choices[0].message.content

## 4. Check every promo against the rules (code)

Line numbers become real timestamps here. Flags:
- `bad_line`: a line number that isn't in the script, or segments out of order
- `too_short` / `too_long`: total length isn't between 20 and 60s

If a promo breaks a rule, the model is told exactly what's wrong and gets **one** retry. Whatever is still broken after that keeps its flag and moves down the ranking. Nothing is silently dropped.

In [ ]:
def check_promo(promo, ep):
    story = story_lines(ep)
    flags = []
    segments = []
    for seg in promo.segments:
        a, b = seg.from_line, seg.to_line
        if 1 <= a <= b <= len(story):
            segments.append((story[a - 1]["start"], story[b - 1]["end"], a, b))
        else:
            flags.append("bad_line")
    segments.sort()
    for prev, cur in zip(segments, segments[1:]):
        if cur[0] < prev[1]:
            flags.append("bad_line")

    if not segments:
        return {"episode": ep["number"], "segments": "", "seg_list": [], "line_list": [], "duration": 0, "hook": "", "cliffhanger": "",
                "why": promo.why, "hook_score": 1, "cliff_score": 1, "standalone_score": 1, "flags": ["bad_line"]}

    duration = sum(e - s for s, e, _, _ in segments)
    if duration < 20:
        flags.append("too_short")
    if duration > 60:
        flags.append("too_long")

    first_line = story[segments[0][2] - 1]
    last_line = story[segments[-1][3] - 1]
    return {
        "episode": ep["number"],
        "segments": " + ".join(f"{mmss(s)}-{mmss(e)}" for s, e, _, _ in segments),
        "seg_list": [(s, e) for s, e, _, _ in segments],
        "line_list": [(a, b) for _, _, a, b in segments],
        "duration": round(duration, 1),
        "hook": line_text(first_line),
        "cliffhanger": line_text(last_line),
        "why": promo.why,
        "hook_score": min(max(promo.hook_score, 1), 10),
        "cliff_score": min(max(promo.cliff_score, 1), 10),
        "standalone_score": min(max(promo.standalone_score, 1), 10),
        "flags": sorted(set(flags)),
    }

def what_is_wrong(raw, ep):
    try:
        promos = EpisodePromos.model_validate_json(raw).promos
    except ValidationError as e:
        return f"That JSON did not match the shape. Error: {e}"
    problems = []
    for i, p in enumerate(promos[:3], start=1):
        row = check_promo(p, ep)
        if "too_long" in row["flags"] or "too_short" in row["flags"]:
            problems.append(f"Promo {i} is {row['duration']} seconds long. It must be 20 to 60 seconds.")
        if "bad_line" in row["flags"]:
            problems.append(f"Promo {i} uses line numbers that don't exist or overlap. Lines go from L1 to L{len(story_lines(ep))}.")
    return "\n".join(problems)

def find_promos(ep):
    raw_path = os.path.join(OUT, "raw", f"episode-{ep['number']:03d}.txt")
    retry_path = raw_path.replace(".txt", "-retry.txt")

    if USE_CACHE and os.path.exists(retry_path):
        raw = open(retry_path, encoding="utf-8").read()
    elif USE_CACHE and os.path.exists(raw_path):
        raw = open(raw_path, encoding="utf-8").read()
    else:
        messages = [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": f"Show: {SHOW}\nEpisode {ep['number']}: {ep['logline']}\n\nScript:\n{shrink(ep)}"},
        ]
        raw = ask_llm(messages)
        open(raw_path, "w", encoding="utf-8").write(raw)
        problem = what_is_wrong(raw, ep)
        if problem:
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": problem + "\nFix this and send the full JSON again."})
            raw = ask_llm(messages)
            open(retry_path, "w", encoding="utf-8").write(raw)

    try:
        return EpisodePromos.model_validate_json(raw).promos[:3]
    except ValidationError:
        print(f"  ep {ep['number']}: model reply still broken, skipping (see {raw_path})")
        return []

fake = Promo(segments=[Segment(from_line=1, to_line=40)], why="test", hook_score=9, cliff_score=9, standalone_score=9)
print(check_promo(fake, episodes[0])["flags"], check_promo(fake, episodes[0])["duration"], "s")
fake = Promo(segments=[Segment(from_line=5, to_line=500)], why="test", hook_score=9, cliff_score=9, standalone_score=9)
print(check_promo(fake, episodes[0])["flags"])

## 5. Run on every episode

In [ ]:
rows = []
retries = 0
t0 = time.time()
for ep in episodes:
    promos = find_promos(ep)
    if os.path.exists(os.path.join(OUT, "raw", f"episode-{ep['number']:03d}-retry.txt")):
        retries += 1
    flagged = 0
    for p in promos:
        row = check_promo(p, ep)
        rows.append(row)
        if row["flags"]:
            flagged += 1
    print(f"ep {ep['number']}: {len(promos)} promos ({flagged} flagged)")
run_seconds = round(time.time() - t0)
print(len(rows), "promos in", run_seconds, "seconds;", retries, "episodes needed a retry")

## 6. Rank (code), then the cold-viewer test (AI)

Code score = hook + cliffhanger + standalone − 5 per flag, +2 for episodes 1 to 10. Early episodes spoil less, and a new viewer who taps through starts at episode 1. If two promos overlap by more than half, only the better one is kept.

The model's own scores are generous, and each episode was scored in a separate call. So the top 20 clean promos get a **cold-viewer test**: a fresh call that sees *only the clip*, with no show name, no logline and no other episodes, just like a stranger on TikTok. It has to say what happens, whether it makes sense and what question the ending leaves. Confusing clips drop out. The final top 10 allows at most 2 promos per episode, so the list has variety.

In [ ]:
df = pd.DataFrame(rows)
df["score"] = (df["hook_score"] + df["cliff_score"] + df["standalone_score"]
               - 5 * df["flags"].apply(len) + 2 * (df["episode"] <= 10))
df = df.sort_values(["score", "episode"], ascending=[False, True]).reset_index(drop=True)

def overlap(a, b):
    total = 0
    for s1, e1 in a:
        for s2, e2 in b:
            total += max(0, min(e1, e2) - max(s1, s2))
    return total

keep = []
for _, row in df.iterrows():
    same_ep = [k for k in keep if k["episode"] == row["episode"]]
    if any(overlap(row["seg_list"], k["seg_list"]) > 0.5 * min(row["duration"], k["duration"]) for k in same_ep):
        continue
    keep.append(row)
ranked = pd.DataFrame(keep).reset_index(drop=True)
ranked["id"] = ranked.index + 1
print(len(df), "promos ->", len(ranked), "after removing overlaps,", (ranked["flags"].apply(len) == 0).sum(), "with no flags")

In [ ]:
class ColdView(BaseModel):
    what_happens: str
    makes_sense: bool
    open_question: str
    stop_scrolling: int
    keep_watching: int

COLD_PROMPT = """You are scrolling TikTok. A clip from a drama series you have never heard of comes up.
You know nothing about the show except what is in this clip. Below is what you see and hear, in order.
Answer honestly, as a normal viewer, in JSON only:
{"what_happens": "one sentence", "makes_sense": true or false (could you follow it with no other context?), "open_question": "the question the ending leaves you wanting answered, or empty if the ending feels finished", "stop_scrolling": 1-10 (would the first 3 seconds make you stop?), "keep_watching": 1-10 (how badly do you want to see what happens next?)}"""

def clip_text(row):
    ep = [e for e in episodes if e["number"] == row["episode"]][0]
    parts = []
    for s, e in row["seg_list"]:
        for line in ep["lines"]:
            if line["start"] >= s - 0.01 and line["end"] <= e + 0.01:
                parts.append(line_text(line))
        parts.append("(cut)")
    parts.pop()
    return "\n".join(parts)

def cold_view(row):
    raw_path = os.path.join(OUT, "raw", safe_name(f"cold-ep{row['episode']:03d}-{row['segments']}") + ".txt")
    if USE_CACHE and os.path.exists(raw_path):
        raw = open(raw_path, encoding="utf-8").read()
    else:
        raw = ask_llm([{"role": "system", "content": COLD_PROMPT},
                       {"role": "user", "content": clip_text(row)}], model=JUDGE_MODEL)
        open(raw_path, "w", encoding="utf-8").write(raw)
    try:
        return ColdView.model_validate_json(raw)
    except ValidationError:
        return None

shortlist = ranked[ranked["flags"].apply(len) == 0].head(20).copy()
views = [cold_view(r) for _, r in shortlist.iterrows()]
shortlist["what_happens"] = [v.what_happens if v else "" for v in views]
shortlist["makes_sense"] = [v.makes_sense if v else False for v in views]
shortlist["open_question"] = [v.open_question if v else "" for v in views]
shortlist["stop_scrolling"] = [min(max(v.stop_scrolling, 1), 10) if v else 1 for v in views]
shortlist["keep_watching"] = [min(max(v.keep_watching, 1), 10) if v else 1 for v in views]
shortlist["final_score"] = (shortlist["score"] + shortlist["stop_scrolling"] + shortlist["keep_watching"]
                            - 10 * (~shortlist["makes_sense"].astype(bool)) - 5 * (shortlist["open_question"].str.strip() == ""))
shortlist = shortlist.sort_values("final_score", ascending=False)

top = []
for _, r in shortlist.iterrows():
    if len(top) < 10 and sum(1 for t in top if t["episode"] == r["episode"]) < 2:
        top.append(r)
top = pd.DataFrame(top).reset_index(drop=True)
top["post_first"] = top.index + 1
top["post_reason"] = [f"Stops the scroll {r['stop_scrolling']}/10, leaves the viewer asking: {r['open_question']}" for _, r in top.iterrows()]
top[["post_first", "id", "episode", "segments", "duration", "hook", "cliffhanger", "open_question", "makes_sense"]]

### Hook second pass (AI picks, code checks)

Reading the first results showed the main weakness: half of the top 10 opened on a *reply* in the middle of an argument ("Not anymore."), when a stronger line was a few seconds earlier or later. So each top-10 promo gets one more call. The model sees the clip plus the 6 lines before it and picks the best opening line. Code keeps the change only if the clip is still 20 to 60s.

In [ ]:
class HookPick(BaseModel):
    start_line: int
    reason: str

HOOK_PROMPT = """You are cutting a promo clip for TikTok. Below are numbered lines: the clip and a few lines before it.
Pick the line the clip should START on, so that the first 3 seconds make a stranger stop scrolling.
Prefer a shocking line, a threat, a slap, a reveal: something that works without hearing the line before it.
Avoid starting on a reply ("Not anymore.", "Why?") that only makes sense after an earlier line.
Reply with JSON only: {"start_line": 12, "reason": "why this line makes a stranger stop scrolling"}"""

def better_hook(row):
    ep = [e for e in episodes if e["number"] == row["episode"]][0]
    story = story_lines(ep)
    a, b = row["line_list"][0]
    lo = max(1, a - 6)
    raw_path = os.path.join(OUT, "raw", safe_name(f"hook-ep{row['episode']:03d}-{row['segments']}") + ".txt")
    if USE_CACHE and os.path.exists(raw_path):
        raw = open(raw_path, encoding="utf-8").read()
    else:
        options = [f"L{n} [{story[n - 1]['start']:.1f}s] {line_text(story[n - 1])}" for n in range(lo, b + 1)]
        raw = ask_llm([{"role": "system", "content": HOOK_PROMPT},
                       {"role": "user", "content": "\n".join(options)}], model=JUDGE_MODEL)
        open(raw_path, "w", encoding="utf-8").write(raw)
    try:
        pick = HookPick.model_validate_json(raw)
    except ValidationError:
        return None
    n = pick.start_line
    if n == a or not lo <= n <= b:
        return None
    new_start = story[n - 1]["start"]
    new_duration = row["duration"] - (new_start - row["seg_list"][0][0])
    if not 20 <= new_duration <= 60:
        return None
    return n, new_start, round(new_duration, 1), pick.reason

new_segs, new_text, new_duration, new_hook, moved = [], [], [], [], []
for _, r in top.iterrows():
    segs, text, duration, hook, note = r["seg_list"], r["segments"], r["duration"], r["hook"], ""
    result = better_hook(r)
    if result is not None:
        n, new_start, duration, reason = result
        ep = [e for e in episodes if e["number"] == r["episode"]][0]
        segs = [(new_start, r["seg_list"][0][1])] + list(r["seg_list"][1:])
        text = " + ".join(f"{mmss(s)}-{mmss(e)}" for s, e in segs)
        hook = line_text(story_lines(ep)[n - 1])
        note = f"start {r['segments'][:8]} -> {mmss(new_start)}: {reason}"
    new_segs.append(segs)
    new_text.append(text)
    new_duration.append(duration)
    new_hook.append(hook)
    moved.append(note)
top["seg_list"] = new_segs
top["segments"] = new_text
top["duration"] = new_duration
top["hook"] = new_hook
top["hook_moved"] = moved
print((top["hook_moved"] != "").sum(), "of", len(top), "hooks moved")
top[["post_first", "episode", "segments", "duration", "hook", "hook_moved"]]

## 7. Save results and quality numbers

In [ ]:
def md_table(d):
    d = d.copy()
    if "flags" in d:
        d["flags"] = d["flags"].apply(lambda f: ", ".join(f) if f else "")
    return d.to_markdown(index=False)

ranked.drop(columns=["seg_list", "line_list"]).to_csv(os.path.join(OUT, "ranked_promos.csv"), index=False)
shortlist.drop(columns=["seg_list", "line_list"]).to_csv(os.path.join(OUT, "cold_viewer_test.csv"), index=False)
top.drop(columns=["seg_list", "line_list"]).to_csv(os.path.join(OUT, "post_first.csv"), index=False)
df.drop(columns=["seg_list", "line_list"]).to_json(os.path.join(OUT, "all_promos.json"), orient="records", indent=2)

with open(os.path.join(OUT, "ranked_promos.md"), "w", encoding="utf-8") as f:
    f.write(f"# Promos for {SHOW}\n\nModel: `{MODEL}`, judge: `{JUDGE_MODEL}`. Times are MM:SS.ss from the start of each episode.\n\n")
    f.write("## Post these first\n\n")
    f.write(md_table(top[["post_first", "episode", "segments", "duration", "hook", "cliffhanger", "why", "post_reason", "hook_moved"]]))
    f.write("\n\n## Cold-viewer test on the shortlist\n\n")
    f.write(md_table(shortlist[["id", "episode", "segments", "what_happens", "makes_sense", "open_question", "stop_scrolling", "keep_watching", "final_score"]]))
    f.write("\n\n## Full ranking (code score, overlaps removed)\n\n")
    f.write(md_table(ranked[["id", "episode", "segments", "duration", "hook", "cliffhanger", "why", "score", "flags"]]))
    f.write("\n")

flag_counts = {}
for fl in df["flags"]:
    for name in fl:
        flag_counts[name] = flag_counts.get(name, 0) + 1

info = {
    "show": SHOW,
    "model": MODEL,
    "judge_model": JUDGE_MODEL,
    "base_url": BASE_URL,
    "episodes": len(episodes),
    "promos_suggested": len(df),
    "promos_clean": int((df["flags"].apply(len) == 0).sum()),
    "episodes_needing_retry": retries,
    "flag_counts": flag_counts,
    "after_overlap_removal": len(ranked),
    "top10_hooks_moved": int((top["hook_moved"] != "").sum()),
    "shortlist_confusing_to_stranger": int((~shortlist["makes_sense"].astype(bool)).sum()),
    "avg_duration": round(df["duration"].mean(), 1),
    "run_seconds_this_time": run_seconds,
    "date": time.strftime("%Y-%m-%d %H:%M"),
}
json.dump(info, open(os.path.join(OUT, "run_info.json"), "w"), indent=2)
print(json.dumps(info, indent=2))

## 8. Human check: read a promo next to the script

An editor should read (or watch) every top pick before it goes out. This prints exactly what the clip covers, CAMERA lines included.

In [ ]:
def show_clip(row_id):
    r = ranked[ranked["id"] == row_id].iloc[0]
    ep = [e for e in episodes if e["number"] == r["episode"]][0]
    print(f"Episode {r['episode']} | {r['segments']} | {r['duration']}s | flags: {r['flags']}")
    print("-" * 60)
    for s, e in r["seg_list"]:
        for line in ep["lines"]:
            if line["start"] >= s - 0.01 and line["end"] <= e + 0.01:
                who = line["speaker"] if line["kind"] == "DIALOGUE" else line["kind"]
                print(f"{mmss(line['start'])}  {who}: {line['text']}")
        print("   ... cut ...")

if len(top):
    show_clip(int(top.iloc[0]["id"]))

## Optional: compare runs with different models

Each model writes to its own folder, so you can run the notebook again with another `MODEL` and compare.

In [ ]:
show_dir = os.path.join("outputs", safe_name(SHOW))
runs = []
for name in sorted(os.listdir(show_dir)):
    path = os.path.join(show_dir, name, "run_info.json")
    if os.path.exists(path):
        runs.append(json.load(open(path)))
pd.DataFrame(runs).reindex(columns=["model", "episodes", "promos_suggested", "promos_clean", "episodes_needing_retry",
                                    "top10_hooks_moved", "shortlist_confusing_to_stranger", "avg_duration", "flag_counts"])